# Long Short-Term Memory

Replication of Hochreiter and Schmidhuber (1997), *Long Short-Term Memory*, Neural
Computation 9(8).

The paper introduces the LSTM cell with its constant-error carousel to overcome the
vanishing-gradient problem that prevents ordinary recurrent networks from learning
long-range dependencies. We reproduce this on a long-dependency classification task: the
label is set by the first timestep, and the network must report it after a long run of
random, uninformative noise inputs. A plain tanh RNN is perturbed by the noise and loses
the early signal across the long gap, whereas the LSTM learns to gate the noise out and
preserve the bit in its cell state (with the standard forget-gate bias initialized to 1).
We show the LSTM reaches near-perfect accuracy while the RNN stays well below it.

In [1]:
import torch, torch.nn as nn
torch.manual_seed(0)

In [2]:
# Label is the first element; positions 1..T-1 are random +/-1 noise (interference).
T = 50
def make_batch(bs):
    bits = torch.randint(0, 2, (bs,))
    noise = (torch.randint(0, 2, (bs, T, 1)).float() * 2 - 1)
    noise[:, 0, 0] = bits.float() * 2 - 1               # plant the signal at t=0
    return noise, bits.long()
xb, yb = make_batch(4)
print("sequence length", T, "| labels (bit to remember):", yb.tolist())

sequence length 50 | labels (bit to remember): [0, 1, 1, 0]


In [3]:
class Recurrent(nn.Module):
    def __init__(self, kind, hidden=64):
        super().__init__()
        self.kind = kind
        self.rnn = (nn.LSTM if kind == "lstm" else nn.RNN)(1, hidden, batch_first=True)
        if kind == "lstm":
            # Standard trick: initialize the forget-gate bias to 1 so the cell retains memory.
            for name, p in self.rnn.named_parameters():
                if "bias" in name:
                    n = p.size(0); p.data[n//4:n//2].fill_(1.0)
        self.fc = nn.Linear(hidden, 2)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1])

def train(kind, steps=4000):
    torch.manual_seed(0)
    net = Recurrent(kind); opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    lf = nn.CrossEntropyLoss()
    for s in range(steps):
        x, y = make_batch(64)
        opt.zero_grad(); lf(net(x), y).backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)   # standard for recurrent nets
        opt.step()
    net.eval(); x, y = make_batch(3000)
    with torch.no_grad():
        acc = (net(x).argmax(1) == y).float().mean().item()
    return acc

In [4]:
rnn_acc  = train("rnn")
lstm_acc = train("lstm")
print(f"plain RNN accuracy (T={T}): {rnn_acc*100:.1f}%  (noise erases the early signal)")
print(f"LSTM accuracy      (T={T}): {lstm_acc*100:.1f}%  (cell state preserves the long dependency)")

plain RNN accuracy (T=50): 47.6%  (noise erases the early signal)
LSTM accuracy      (T=50): 100.0%  (cell state preserves the long dependency)
